> 在线讲义：[第 13 章 · 实时执行：RTC，边想下一段，边把这一段走完](https://wikieden.github.io/vla-from-scratch/lessons/13-rtc.html) · 本书《VLA 从零到 π0》

# 第 13 章 · 实时执行：RTC，边想下一段，边把这一段走完

配套讲义：仓库里 `vla-deep-dive/notes/lessons/13-rtc.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `delay_steps(80, 20)` | d = 4 |
| 第 3 步 | `soft_mask(16, 4, 5)` | 1, 1, 1, 1, 0.712, 0.488, 0.316, 0.189, 0.099, 0.041, 0.010, 0 … |

H = 16、d = 4、s = 5 取自 real-time-chunking-2506.07339.txt 图 3。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math

**函数 `delay_steps`**：d = ⌊δ / Δt⌋：想一次的工夫，机器人本该走几步。

In [ ]:
def delay_steps(latency_ms, period_ms):
    """d = ⌊δ / Δt⌋：想一次的工夫，机器人本该走几步。"""
    return math.floor(latency_ms / period_ms)

**函数 `soft_mask`**：式 5：前 d 步权重 1；中间区 c(e^c − 1)/(e − 1)；最后 s 步 0。

In [ ]:
def soft_mask(h, d, s):
    """式 5：前 d 步权重 1；中间区 c(e^c − 1)/(e − 1)；最后 s 步 0。"""
    w = []
    for i in range(h):
        if i < d:
            w.append(1.0)
        elif i < h - s:
            c = (h - s - i) / (h - s - d + 1)
            w.append(c * (math.exp(c) - 1) / (math.e - 1))
        else:
            w.append(0.0)
    return w

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 推理延迟折成控制步")
print(f"  50 Hz（Δt = 20 ms）、想一次 80 ms：d = {delay_steps(80, 20)}")

In [ ]:
print("\n第三步 软遮罩权重（H = 16、d = 4、s = 5）")
w = soft_mask(16, 4, 5)
print("  " + " ".join(f"{x:.3f}" for x in w))

## 3. 代码里才看得到的

1. `soft_mask` 的 c 分母是 H − s − d + 1，所以中间区第一步的 c 是 7/8 而不是 1：权重从 1 往下有一个台阶（1 → 0.712）。
2. LeRobot 的 `_linweights` 用 `linspace(1, 0, n + 2)[1:-1]` 生成 c，和式 5 的写法不同但思路一样，数值在中间几步会有小差别。

## 4. 自己改一改

- 把 s 从 5 改成 8，看中间区变短后权重怎么变。
- 把指数形状去掉（只用 c），比较哪种下降得更快。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 说着这句想下句：停了冷场，硬接跑题。
> 走掉的冻住，剩下的照着补。
> 近的听旧计划，远的听新画面。
> 不重训，加引导，慢模型也能流畅。